## Problem N: [Problem 1: Representing SHA-256 Data]


### Definition
A 32-bit word is a number made up of exactly 32 bits, so it can hold any
value from 0 up to 4,294,967,295. SHA-256 uses these 32-bit words
throughout its calculations. If a calculation would go above this maximum,
the result wraps back around to 0 instead of growing bigger.

### Research
To refresh my understanding of hashing before starting this assessment, I
looked up some information on [GeeksforGeeks](https://www.geeksforgeeks.org/computer-networks/sha-256-and-sha-3/),
covering how hashing works in general and the different types of hashing
algorithms that exist.

For this specific problem, I also looked into how Python represents numbers.
Python's built-in `int` type has no fixed size — it just keeps growing as
needed, with no limit. This doesn't match what SHA-256 requires, since the
standard expects every value to stay within 32 bits and wrap around when it
goes over. NumPy's `uint32` type behaves this way, making it a good choice
for representing 32-bit words in this implementation. 

### Implementation
Below, a 32-bit word is represented using `numpy.uint32`.

In [32]:
import numpy as np

# A 32-bit word can hold values from 0 to 2**32 - 1
word = np.uint32(4294967295)
print(word)

4294967295


### Testing
Demonstrating that `numpy.uint32` wraps around on overflow, rather than
growing like a native Python `int`.

In [33]:
# NumPy uint32 wraps around at 2**32
wrapped = np.uint32(4294967295) + np.uint32(1)
print(wrapped)  # expect 0

# Compare to plain Python int, which just keeps growing
plain = 4294967295 + 1
print(plain)  # expect 4294967296

0
4294967296


C:\Users\Bogdan Nwachukwu\AppData\Local\Temp\ipykernel_3688\2786519109.py:2: RuntimeWarning: overflow encountered in scalar add
  wrapped = np.uint32(4294967295) + np.uint32(1)


### Summary
A 32-bit word is represented using `numpy.uint32`, which enforces the fixed
width and wraparound behavior required for correct SHA-256 arithmetic.

### Definition
A sequence of 32-bit words is simply a list of several 32-bit numbers,
kept together in order. SHA-256 needs this for things like the eight
starting hash values and the 64 values used in each round of processing.

### Research
A plain Python `list` can hold multiple numbers, but it doesn't enforce
that every value stays within 32 bits — it would happily hold a mix of
small numbers, huge numbers, or even non-numbers. NumPy arrays, however,
can be given a fixed `dtype` (data type), so every single value in the
array is forced to be a proper 32-bit unsigned integer, matching what
SHA-256 requires throughout.

### Implementation
Below, a sequence of 32-bit words is represented using a NumPy array with
`dtype=np.uint32`.

In [34]:
import numpy as np

# A sequence of 32-bit words, stored as a NumPy array
words = np.array([1, 2, 3, 4294967295], dtype=np.uint32)
print(words)
print(words.dtype)

[         1          2          3 4294967295]
uint32


### Testing
Confirming that every value in the array is forced to stay within the
32-bit range, wrapping around on overflow just like a single word does.

In [35]:
# Adding 1 to the last word should wrap it around to 0
words[-1] = words[-1] + np.uint32(1)
print(words)

[1 2 3 0]


C:\Users\Bogdan Nwachukwu\AppData\Local\Temp\ipykernel_3688\4183265283.py:2: RuntimeWarning: overflow encountered in scalar add
  words[-1] = words[-1] + np.uint32(1)


### Summary
A sequence of 32-bit words is represented using a NumPy array with
dtype=np.uint32, which keeps every element fixed-width and consistent
with the individual word representation used above.

### Definition
The input message is the original data someone wants to hash — for
example, a piece of text. SHA-256 doesn't work on text directly; it
works on raw bytes, so the message needs to be represented as a
sequence of bytes before any hashing can begin.

### Research
In Python, text is normally stored as a `str`, but `str` represents
characters, not raw binary data, and can't be directly processed as
bits and bytes the way SHA-256 requires. Python's built-in `bytes`
type is the natural fit instead - it's an immutable sequence of raw
byte values (0–255), exactly matching how the Secure Hash Standard
describes a message. Any text first needs to be *encoded* (e.g. using
UTF-8) to convert it into this `bytes` form.

### Implementation
Below, an input message is represented using Python's built-in `bytes`
type, created by encoding a text string.

In [36]:
# Representing an input message as bytes
message = "hello".encode("utf-8")
print(message)
print(type(message))

b'hello'
<class 'bytes'>


### Testing
Confirming that different input strings, including non-ASCII characters,
correctly convert into a bytes object.

In [37]:
# Test with a simple ASCII message
msg1 = "hello".encode("utf-8")
print(msg1)

# Test with an empty message
msg2 = "".encode("utf-8")
print(msg2)

# Test with a non-ASCII character
msg3 = "héllo".encode("utf-8")
print(msg3)

b'hello'
b''
b'h\xc3\xa9llo'


### Summary
An input message is represented using Python's built-in bytes type,
obtained by encoding text (e.g. with UTF-8). This provides the raw
byte sequence that SHA-256 operates on.

# End of Notebook